# Notebook 05 · L3：超图逻辑依赖（AND / OR / k-of-N）

**L3 = L2 + 超边（功能单元）+ 逻辑激活规则**，这是研究内容 1 里最值钱的一级：
L0–L2 文献里都有现成做法，**高阶逻辑依赖是本项目要做的活**。

- 超边：一条边连多个节点（不只是一对）——`Hyperedge(members, rule, k)`
- 规则：**AND**（全活才行）/ **OR**（任一活着就行）/ **k-of-N**（降级运行）
- 功能度量 `F(t)` = 活跃功能单元占比 —— 从"结构连通"升级到"业务可用"

⭐ 退化性检验：只用二元 AND 超边（等价成对耦合）时必须与 L0 完全一致。

## 环境准备（每个 notebook 都要先跑这两格）

In [ ]:
"""环境准备：定位项目根目录、加 src 到搜索路径、设好画图参数。"""
import sys
from pathlib import Path

# 从当前工作目录往上找，直到看见 src/phase6_interdependent（这样在 notebooks/ 里也能跑）
root = Path.cwd()
while not (root / "src" / "phase6_interdependent").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

RESULTS = root / "results"
RESULTS.mkdir(exist_ok=True)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("项目根目录：", root)

In [ ]:
# 公共底座：一对 BA 双层网络 + 骨架类（后面所有 notebook 都从这一格开始）
from phase6_interdependent.core import InterdependentNetwork

N, m = 300, 3
G_A = nx.barabasi_albert_graph(N, m, seed=42)   # 层 A：比如电力网
G_B = nx.barabasi_albert_graph(N, m, seed=43)   # 层 B：比如通信网
nodes_A = sorted(G_A.nodes())
print(f"双层网络就绪：N={N}, 每层边数={G_A.number_of_edges()}")

## 第 1 步

In [ ]:
from phase6_interdependent.L3_hypergraph_logic import (
    Hyperedge, HyperedgeNetwork, pair_hyperedges,
    random_functional_hyperedges, functional_measure
)

# 1) 单条超边的三种语义（手工算一遍，理解规则）
alive = {"A": {0: True, 1: True, 2: False}, "B": {0: True}}
members = [("A", 0), ("A", 1), ("A", 2)]
for rule, k in [("AND", 1), ("OR", 1), ("k-of-N", 2), ("k-of-N", 3)]:
    e = Hyperedge(members, rule=rule, k=k)
    print(f"{rule:<7} k={k} → 活跃？{e.is_active(alive)}")

## 第 2 步

In [ ]:
# 2) 退化性检验：二元 AND 超边（等价层间耦合）必须与 L0 一致
from phase6_interdependent.L0_static_dependency import percolation_curve

fractions = np.linspace(0.0, 0.3, 7)
rng = np.random.default_rng(42)
nodes_A = sorted(G_A.nodes())
S_L0, S_L3 = [], []
for f in fractions:
    k = int(round(len(nodes_A) * f))
    a, b = [], []
    for _ in range(3):
        failed = ([] if k == 0 else rng.choice(nodes_A, size=k, replace=False))
        n0 = InterdependentNetwork(G_A, G_B, seed=42)
        n3 = HyperedgeNetwork(G_A, G_B, seed=42)      # 默认 = 层间成对 AND 超边
        a.append(n0.run(failed)["S_A"])
        b.append(n3.run(failed, rule=n3.hyperedge_rule())["S_A"])
    S_L0.append(np.mean(a)); S_L3.append(np.mean(b))

assert np.allclose(S_L3, S_L0, atol=1e-9), "退化性检验失败！"
print("✅ 退化性检验通过：二元 AND 超边时 L3 ≡ L0")

## 第 3 步

In [ ]:
# 3) 真实功能单元（3 个节点共同支撑一个单元）下的 AND / k-of-N / OR
template = HyperedgeNetwork(G_A, G_B, seed=42)
units = random_functional_hyperedges(template, n_edges=60, size=3, rule="AND", seed=42)
settings = {
    "AND": [Hyperedge(list(e.members), "AND") for e in units],
    "k-of-N(k=2)": [Hyperedge(list(e.members), "k-of-N", k=2) for e in units],
    "OR": [Hyperedge(list(e.members), "OR") for e in units],
}

plt.figure(figsize=(8, 5))
for label, edges in settings.items():
    rng = np.random.default_rng(42)
    curve = []
    for f in fractions:
        k = int(round(len(nodes_A) * f))
        trials = []
        for _ in range(3):
            failed = ([] if k == 0 else rng.choice(nodes_A, size=k, replace=False))
            n = HyperedgeNetwork(G_A, G_B, hyperedges=edges, seed=42)
            n.run(failed, rule=n.hyperedge_rule())
            trials.append(n.functional_measure())
        curve.append(np.mean(trials))
    plt.plot(fractions, curve, "o-", label=f"{label} 语义")
plt.xlabel("移除比例 f"); plt.ylabel("功能度量 F（活跃功能单元占比）")
plt.title("L3：功能单元的逻辑语义决定韧性")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig(RESULTS / "notebook_05_L3_semantics.png", dpi=150, bbox_inches="tight")
plt.show()

**思考题**

1. 制造网络里，"AND 单元"和"OR 单元"分别对应什么现实结构？
2. 为什么"功能度量 F"比"存活节点比例 S"更贴近业务结论？
3. 超边是否可以跨层（A 层节点 + B 层节点组成一个功能单元）？怎么改代码？